## accounts_school 학교 테이블 전처리

In [1]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [2]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_school`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

   id   address  student_count school_type
0   4  충청북도 충주시            239           H
1   6  충청북도 충주시            200           H
2   7  충청북도 충주시            114           H
3  13  충청북도 충주시             80           H
4  16  충청북도 충주시            143           H


## 결측치 확인 및 데이터 정보 확인

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5951 entries, 0 to 5950
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             5951 non-null   Int64
 1   address        5951 non-null   str  
 2   student_count  5951 non-null   Int64
 3   school_type    5951 non-null   str  
dtypes: Int64(2), str(2)
memory usage: 342.1 KB


- 결측치 없음
- id, student_count는 수치형
- address, school_type는 문자형

## 중복값 체크

In [4]:
df.duplicated().sum()

df['id'].duplicated().sum()

np.int64(0)

- 중복 데이터 없음

## 이상치

In [5]:
df.describe()

,id,student_count
count,5951.0,5951.0
mean,2981.026046,113.772979
std,1719.08402,102.863428
min,4.0,0.0
25%,1493.5,16.0
50%,2981.0,97.0
75%,4469.5,183.0
max,5965.0,578.0


In [6]:
df['school_type'].unique()

<ArrowStringArray>
['H', 'M']
Length: 2, dtype: str

In [8]:
df.sort_values(by='student_count').head(20)

,id,address,student_count,school_type
4198,2886,부산광역시 기장군,0,H
4253,4583,경기도 평택시,0,H
4254,4648,경기도 파주시,0,H
4255,4661,경기도 파주시,0,H
4256,4671,경기도 이천시,0,H
4257,4685,경기도 이천시,0,H
4258,4700,경기도 의정부시,0,H
4259,4724,경기도 의정부시,0,H
4260,4735,경기도 의왕시,0,H
4261,4752,경기도 용인시 처인구,0,H


# (추가 전처리 진행)

* 분석 과정에서 일부 주소 표기 및 데이터 간 싱크 문제가 확인되어, 정확한 분석을 위해 추가적인 데이터 확인 및 전처리를 진행하고자 함.

## 주소 값이 존재하지 않는 데이터 존재

In [6]:
df[df['address'] == '-']

,id,address,student_count,school_type
4550,5949,-,1,H
4551,5964,-,1,H
4714,5948,-,2,H


* `5949`, `5948` 학교는 `nearbyschool` 전처리 과정에서 거리 이상치가 확인되어 이상 학교 또는 테스트 학교로 판단, 삭제 진행
* `5964` 학교는 전처리 과정에서 이상 학교로 판단되지 않았으나, 주소 정보가 누락되어 인근 학교의 주소를 확인하여 주소 복원 시도

In [12]:
temp_sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_nearbyschool` AS ns
    INNER JOIN `{PROJECT_ID}.{DATA_SET}.accounts_school` AS sch
    ON ns.nearby_school_id = sch.id
    WHERE ns.school_id = 5964
"""

temp_df = client.query(temp_sql).to_dataframe()

print(temp_df.head(30))

       id  distance  nearby_school_id  school_id  id_1      address  \
0  178491  0.039154              4513       5964  4513      경기도 화성시   
1  178496  0.055266              4514       5964  4514      경기도 화성시   
2  178498  0.058339              4583       5964  4583      경기도 평택시   
3  178497  0.055511              4585       5964  4585      경기도 평택시   
4  189042  0.079570              4751       5964  4751  경기도 용인시 처인구   
5  178495  0.054407              4764       5964  4764  경기도 용인시 처인구   
6  178494  0.053842              4832       5964  4832      경기도 오산시   
7  178493  0.052513              4833       5964  4833      경기도 오산시   
8  178492  0.044422              4840       5964  4840      경기도 오산시   
9  178499  0.069071              4844       5964  4844      경기도 오산시   

   student_count school_type  
0            348           H  
1              0           H  
2              0           H  
3            115           H  
4             83           H  
5            238           H  
6

In [13]:
temp_sql = f"""
    SELECT * FROM `sns-analysis-prj.sns_analysis.accounts_group`
    WHERE school_id = 5964
"""

temp_df = client.query(temp_sql).to_dataframe()

print(temp_df.head())

      id  grade  class_num  school_id
0  83939      1          1       5964
1  83938      2          2       5964


In [14]:
temp_sql = f"""
    SELECT * FROM `sns-analysis-prj.sns_analysis.accounts_user`
    WHERE group_id IN (83939, 83938)
"""

temp_df = client.query(temp_sql).to_dataframe()

print(temp_df.head())

Empty DataFrame
Columns: [id, is_superuser, is_staff, gender, point, friend_id_list, is_push_on, created_at, block_user_id_list, hide_user_id_list, ban_status, report_count, alarm_count, pending_chat, pending_votes, group_id]
Index: []


* 인근 학교의 주소를 추가로 확인하였으나, 특정 지역에 소속된 학교가 아니어서 해당 학교의 주소를 추정하기 어렵다고 판단
* 해당 학교 소속 학생이 1명으로 매우 적어 분석에 미치는 영향이 미미할 것으로 판단되고
* 또한 분석 대상 학생 중 해당 학교 소속 그룹에 포함된 학생이 없어 삭제 진행

## 주소 형태가 동일하지 않은 문제 발견

In [15]:
df['address'].unique()

<ArrowStringArray>
[    '충청북도 충주시', '충청북도 청주시 흥덕구', '충청북도 청주시 청원구', '충청북도 청주시 서원구',
 '충청북도 청주시 상당구',     '충청북도 진천군',     '충청북도 증평군',     '충청북도 제천시',
     '충청북도 음성군',     '충청북도 옥천군',
 ...
       '강원 정선군',   '경남 창원시 진해구',      '제주 서귀포시', '경남 창원시 마산합포구',
      '서울 동대문구',       '서울 마포구',     '경상북도 울릉군',   '충남 천안시 동남구',
       '경북 김천시',   '경기 용인시 수지구']
Length: 278, dtype: str

In [16]:
# 1. 주소의 첫 단어(시/도)만 추출해서 어떤 표기들이 있는지 확인
df['sido'] = df['address'].str.split().str[0]
df['sido'].value_counts()

sido
경기도        1195
서울특별시       761
경상남도        474
경상북도        466
전라남도        410
충청남도        366
전라북도        353
부산광역시       331
인천광역시       286
강원도         284
대구광역시       233
충청북도        227
광주광역시       168
대전광역시       160
울산광역시       126
제주특별자치도      80
경기            6
서울            5
경남            4
경북            3
충남            3
-             3
전북            2
대한민국          1
대구            1
인천            1
강원            1
제주            1
Name: count, dtype: int64

* 일부 행정구역명이 축약된 형태로 표기되어 있음
* 대한민국 관련 부분은 정확성 여부를 별도로 확인할 필요가 있음

In [19]:
df[df['sido'] == '대한민국']

,id,address,student_count,school_type,sido
1049,2941,대한민국 강원도 철원군,54,H,대한민국


In [25]:
df[df['sido'].isin(['대한민국', '경기', '서울', '경남', '경북', '충남', '전북', '대구', '인천', '강원', '제주'])]

,id,address,student_count,school_type,sido
201,625,전북 정읍시,102,H,전북
1049,2941,대한민국 강원도 철원군,54,H,대한민국
1203,3335,대구 수성구,72,H,대구
1555,4444,경북 영양군,86,H,경북
2075,5933,충남 아산시,229,H,충남
2076,5934,경기 용인시 기흥구,52,H,경기
2080,5955,경남 창원시 의창구,214,H,경남
2602,1674,인천 남동구,222,M,인천
2896,2610,서울 노원구,73,M,서울
3528,4447,경남 진주시,56,M,경남


* 대한민국이 표기된 데이터는 강원도의 학교를 나타내고 있음
* 이외 행정구역명이 축약된 형태를 가지고 있으므로, 이에 대한 값 대체 진행

# 쿼리 수행

In [ ]:
del_sql = f"""
DELETE FROM `{PROJECT_ID}.{DATA_SET}.accounts_school`
WHERE address = '-'
"""

query_job = client.query(del_sql)
query_job.result()  # DML 완료 대기

print(f"\n삭제 완료! 총 삭제된 행 수: {query_job.num_dml_affected_rows}건")


수정 완료! 총 수정된 행 수: 3건


In [27]:
update_sql = f"""
UPDATE `{PROJECT_ID}.{DATA_SET}.accounts_school`
SET address = CASE
    WHEN address LIKE '대한민국 %' THEN REPLACE(address, '대한민국 ', '')
    WHEN address LIKE '경기 %' THEN REPLACE(address, '경기 ', '경기도 ')
    WHEN address LIKE '서울 %' THEN REPLACE(address, '서울 ', '서울특별시 ')
    WHEN address LIKE '경남 %' THEN REPLACE(address, '경남 ', '경상남도 ')
    WHEN address LIKE '경북 %' THEN REPLACE(address, '경북 ', '경상북도 ')
    WHEN address LIKE '충남 %' THEN REPLACE(address, '충남 ', '충청남도 ')
    WHEN address LIKE '전북 %' THEN REPLACE(address, '전북 ', '전라북도 ')
    WHEN address LIKE '대구 %' THEN REPLACE(address, '대구 ', '대구광역시 ')
    WHEN address LIKE '인천 %' THEN REPLACE(address, '인천 ', '인천광역시 ')
    WHEN address LIKE '강원 %' THEN REPLACE(address, '강원 ', '강원도 ')
    WHEN address LIKE '제주 %' THEN REPLACE(address, '제주 ', '제주특별자치도 ')
    ELSE address
END
WHERE address IS NOT NULL;
"""

query_job = client.query(update_sql)
query_job.result()  # DML 완료 대기

print(f"\n수정 완료! 총 수정된 행 수: {query_job.num_dml_affected_rows}건")


수정 완료! 총 수정된 행 수: 5948건
